In [ ]:
# you modify the following in the command line interface, like
# papermill fit-efficientnet-b0.ipynb test.ipynb -p height 224 -p width 224

### notebook config ###

table_path = None # where your metadata is (necessary)
output_folder = None # where to store your results
model_path = None # where to store your model
num_epochs = None
batch_size = None
num_workers = None # speeds up audio processing
freeze_inner = None # true means only update output layer weights

### AudioDataset override knobs ###

base_folder = None # where high-level acoustics data files lie
height = None # height of spectrogram image
width = None # width of spectrogram image
duration = None # seconds in clip
p_augment = None # probability of training data augmentation
num_augments = None # number of augmentation loops

### Spectrogram augmentation knobs ###

spec_min_gain = None
spec_max_gain = None
spec_max_shift_pct = None
spec_max_time_mask_pct = None
spec_max_time_mask_num = None
spec_max_freq_mask_pct = None
spec_max_freq_mask_num = None

spec_train_p_gain = None
spec_train_p_shift = None
spec_train_p_time_mask = None
spec_train_p_freq_mask = None

spec_val_p_gain = None
spec_val_p_shift = None
spec_val_p_time_mask = None
spec_val_p_freq_mask = None

### don't change these ###

n_samples = None # downsample for software dev purposes
min_gain_db = None
max_gain_db = None
fmin = None
fmax = None
sr = None # standardized sampling rate

In [ ]:
# defaults
if fmax is None:
    from osprey import fmax
if fmin is None:
    from osprey import fmin
if height is None:
    from osprey import height
if width is None:
    from osprey import width
if duration is None:
    from osprey import duration
if base_folder is None:
    from osprey import base_folder
if sr is None:
    from osprey import sr
if p_augment is None:
    from osprey import p_augment
if min_gain_db is None:
    from osprey import min_gain_db
if max_gain_db is None:
    from osprey import max_gain_db
if num_augments is None:
    num_augments = 1
assert num_augments >= 1
num_augments = int(num_augments)

collection_map = {
    'iNat': 'birdclef-2026/train_audio',
    'XC': 'birdclef-2026/train_audio',
    'esc': 'ESC-50-master/audio',
    'arca23k': 'ARCA23K/ARCA23K.audio/ARCA23K.audio',
    'urban8k': 'UrbanSound8K/audio/foldall',
    'dbr': 'dbr-dataset/dataset/dograin',
    'freesound': 'freesound/outside',
}

# override AudioDataset defaults
dataset_sr = sr

if num_epochs is None:
    num_epochs = 2
if batch_size is None:
    batch_size = 2 ** 7
if num_workers is None:
    num_workers = 0
if n_samples is None:
    n_samples = 1e9
if freeze_inner is None:
    freeze_inner = False
if output_folder is None:
    output_folder = '../results'
loss_folder = f"{output_folder}/loss"
metrics_folder = f"{output_folder}/metrics"

In [ ]:
# trigger cli failure
assert table_path is not None, 'Must provide an audio metadata file'

## Importing packages and other setup

In [ ]:
# common imports
import os
import time
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, precision_recall_fscore_support, f1_score

# audio libraries
import IPython.display as ipd

# modeling imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import transforms
import torchvision.models as models
from safetensors.torch import save_file
import json

# osprey package
from osprey import (
    SpectrogramDatasetGPU,
    SimpleCNN,
    clean_row,
    get_audio,
    get_mel,
    reformat_image,
    augmenter_spectrogram,
    # keep the CPU dataset imports available if you still need them
    SpectrogramDataset,
    )

# documentation
import numpy.typing as npt
from typing import Optional, Union

os.makedirs(output_folder, exist_ok=True)
os.makedirs(loss_folder, exist_ok=True)
os.makedirs(metrics_folder, exist_ok=True)

# Check if a GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU instead.")

## Loading and preparing the data

In [ ]:
# only birdclef-2026 data
meta = pd.read_csv(table_path)
meta = meta[meta['collection'].isin(['XC','iNat'])]

# subsets
train_data = meta[meta['dataset']=='train'].copy()
le = LabelEncoder() # label encoder
le.fit(train_data['primary_label'])
val_data = meta[meta['dataset']=='validate'].copy()
test_data = meta[meta['dataset']=='test'].copy()

# reset indices
train_data.reset_index(inplace=True, drop=True)
val_data.reset_index(inplace=True, drop=True)
test_data.reset_index(inplace=True, drop=True)

# unique species in datasets
train_species = np.array(train_data.primary_label.unique())
val_species = np.array(val_data.primary_label.unique())
test_species = np.array(train_data.primary_label.unique())

In [ ]:
# smaller subset: 8 batches of 32 samples = 256
num_classes = len(le.classes_)

# sample from test split (use .head(n_samples) instead of .sample(...) if you want deterministic order)
train_data_small = train_data.sample(
    n=min(n_samples, len(train_data)), 
    random_state=42).reset_index(drop=True)
val_data_small = val_data.sample(
    n=min(n_samples, len(val_data)), 
    random_state=42).reset_index(drop=True)

# optional: overwrite active dataset/dataloader to use the small subset
dataset_train = SpectrogramDatasetGPU(
    train_data_small,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
 )
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, num_workers=num_workers, shuffle=False)
dataset_val = SpectrogramDatasetGPU(
    val_data_small,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
 )
dataloader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=num_workers, shuffle=False)

print("Number of training samples:", len(train_data_small))
print("Number of batches:", len(dataloader_train))

In [ ]:
# spectrogram augmentation settings
if spec_min_gain is None:
    spec_min_gain = 5.
if spec_max_gain is None:
    spec_max_gain = 25.
if spec_max_shift_pct is None:
    spec_max_shift_pct = 0.20
if spec_max_time_mask_pct is None:
    spec_max_time_mask_pct = 0.02
if spec_max_time_mask_num is None:
    spec_max_time_mask_num = 5
if spec_max_freq_mask_pct is None:
    spec_max_freq_mask_pct = 0.02
if spec_max_freq_mask_num is None:
    spec_max_freq_mask_num = 5

if spec_train_p_gain is None:
    spec_train_p_gain = p_augment
if spec_train_p_shift is None:
    spec_train_p_shift = p_augment
if spec_train_p_time_mask is None:
    spec_train_p_time_mask = p_augment
if spec_train_p_freq_mask is None:
    spec_train_p_freq_mask = p_augment

def apply_spectrogram_augmentation(
    x,
    *,
    p_gain,
    p_shift,
    p_time_mask,
    p_freq_mask,
 ):
    return augmenter_spectrogram(
        x,
        min_gain=spec_min_gain,
        max_gain=spec_max_gain,
        p_gain=p_gain,
        max_shift_pct=spec_max_shift_pct,
        p_shift=p_shift,
        max_time_mask_pct=spec_max_time_mask_pct,
        max_time_mask_num=spec_max_time_mask_num,
        p_time_mask=p_time_mask,
        max_freq_mask_pct=spec_max_freq_mask_pct,
        max_freq_mask_num=spec_max_freq_mask_num,
        p_freq_mask=p_freq_mask,
    )

## Model definition

In [ ]:
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, num_classes),
)

# criterion = nn.BCEWithLogitsLoss()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model.to(device)

print()

In [ ]:
# total parameter counts
total_params = sum(p.numel() for p in model.parameters())

if freeze_inner:

    # freeze all but the last layer
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

# trainable parameter counts
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# printing summary
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {total_params - trainable_params:,}")

## Training loop

In [ ]:
image_size = (height, width)
channel_size = 3

ft = open(f"{loss_folder}/training_loss.csv", 'w')
fv = open(f"{loss_folder}/validation_loss.csv", 'w')
ft.write('epoch,loss\n')
fv.write('epoch,loss\n')

for _ in range(1, num_epochs + 1):

    # learning
    model.train()
    loss_total = 0
    itr = 0
    for a in range(1, num_augments + 1):
        curr_time = time.time()
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            if (itr % 10) == 0:
                print(f'{itr/len(dataloader_train)*100:.4f}%')
            optimizer.zero_grad()
            X = X.to(device)
            X = apply_spectrogram_augmentation(
                X,
                p_gain=spec_train_p_gain,
                p_shift=spec_train_p_shift,
                p_time_mask=spec_train_p_time_mask,
                p_freq_mask=spec_train_p_freq_mask,
            )
            Z = reformat_image(X, 
                            image_size=image_size, 
                            channel_size=channel_size,
                            )
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
            loss.backward()
            optimizer.step()
        print(f'Epoch {_}, Round {a} took {time.time() - curr_time:0f} seconds')
    print('')

    # inference
    model.eval()
    with torch.no_grad():

        # training loss tracking
        loss_total = 0
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            X = X.to(device)
            Z = reformat_image(X, 
                           image_size=image_size, 
                           channel_size=channel_size,
                           )
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
        print(f'The training loss at epoch {_} is {loss_total / itr:6f}')
        ft.write(f"{_},{loss_total / itr:4f}\n")

        # validation loss tracking
        loss_total = 0
        itr = 0
        for X, Y in dataloader_val:
            itr += 1
            X = X.to(device)
            Z = reformat_image(X, 
                           image_size=image_size, 
                           channel_size=channel_size,
                           )
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
        print(f'The validation loss at epoch {_} is {loss_total / itr:6f}')
        fv.write(f"{_},{loss_total / itr:4f}\n")
    
    print('')

ft.close()
fv.close()

## Save model

In [ ]:
if model_path is not None:

    checkpoint_path = f"{output_folder}/{model_path}"

    tensor_state_dict = model.state_dict()

    metadata = {
        "optimizer_state_dict": json.dumps(optimizer.state_dict(), default=str),
        "num_classes": str(num_classes),
        "label_classes": json.dumps(le.classes_.tolist()),
        "image_size": json.dumps(image_size),
        "channel_size": str(channel_size),
        "epoch": str(num_epochs),
    }

    save_file(tensor_state_dict, checkpoint_path, metadata=metadata)

    print(f"Checkpoint saved safely to: {checkpoint_path}")

## Evaluation loop

In [ ]:
# compute some summary metrics

zipped_data = [
    (train_species, dataloader_train, 'training'),
    (val_species, dataloader_val, 'validation'),
]

for species, dataloader, split_name in zipped_data:

    f0 = open(f"{metrics_folder}/{split_name}_metrics.csv",'w')
    f0.write('category,loss,precision,recall,f1_score\n')

    loss_total = 0.0
    n_batches = 0
    dataloader = dataloader_train
    species = train_species

    model.eval()

    # Initialize empty lists to hold batch tensors
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for X, Y_batch in dataloader:
            # Move everything to device immediately
            X = X.to(device)
            Z_batch = reformat_image(X, 
                                    image_size=image_size, 
                                    channel_size=channel_size)
            Y_batch = Y_batch.to(device) 
            
            logits = model(Z_batch)
            loss = criterion(logits, Y_batch) # Keep loss on GPU

            loss_total += loss.item()
            n_batches += 1

            # Store the batch tensors directly
            y_true_all.append(Y_batch)
            y_pred_all.append(logits.argmax(dim=1))

    # Concatenate all batches at once and move to CPU only once
    y_true = torch.cat(y_true_all).cpu().numpy()
    y_pred = torch.cat(y_pred_all).cpu().numpy()

    # write metrics
    f0.write('overall,')
    f0.write(f"{loss_total/n_batches:.4f},")
    f0.write(f"{precision_score(y_true, y_pred, average='macro', zero_division=0):.4f},")
    f0.write(f"{recall_score(y_true, y_pred, average='macro', zero_division=0):.4f},")
    f0.write(f"{f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    f0.write('\n')

    # metrics for every class
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        s = le.inverse_transform([i])[0]
        f0.write(f"{s},0.,{p:.4f},{r:.4f},{f:.4f}\n")

    f0.close()